# Загрузка переменных окружения

In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [2]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds'

JSON_EXTENSION = "fintoc4.json"

In [3]:
if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())
FILE_PATHS.sort()
FILE_PATHS = FILE_PATHS[:10]

fintoc работает с такой структурой:
```json
[
{"text": str,
"id": 0, 1, 2, ...,
"page": 1, 2, ...,
"depth": 1, 2, ...},
    ...
]
```

In [4]:
import json
from typing import Dict

from pagerlib.file_input import FileInput
from pagerlib.extractors.document_extractor import LogicalStructureExtractor
from pagerlib.extractors.page_extractor import PDFIMGExtractor, Rows2Regions
from pagerlib.dtypes import Section
from pagerlib.extractors.page_extractor.rows2regions.rows2region_glam_20260826 import get_load_model

fi = FileInput()
img_extr = PDFIMGExtractor()
rows2regions = Rows2Regions()
rows2regions.model = get_load_model(path='fine-tuned.pt')
regions2tree = LogicalStructureExtractor()

def pagerlib_pdf2toc(file_path):
    prdf = fi(file_path)
    img_extr.extract(prdf)
    rows2regions.extract(prdf)
    regions2tree.extract(prdf)
    return prdf.data['toc']

def get_header(sec):
    headers = []
    children = sec.children
    for child in children:
        if isinstance(child, Section):
            reg = child.title
            reg.data['level'] = child.level
            headers.append(reg)
            headers.extend(get_header(child))
    return headers


def toc2fintoc_json(toc) -> Dict:
    header_regs = get_header(toc)
    fintoc_json = []
    for id_node, node in enumerate(header_regs):
        reg = {
            "text": node.text,
            "id": id_node,
            "page": node.data['page_num'] + 1, # В PageR специально оставил нумерацию с нуля (поскольку номер определяется не порядком, а пишится на странице и это еще не реализовано)
            "depth": node.data['level']
        }
        fintoc_json.append(reg)
    return fintoc_json    

/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [5]:
toc = pagerlib_pdf2toc(FILE_PATHS[8])
rez = toc2fintoc_json(toc)

/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/utils/_spmm.py:71: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  src = src.to_sparse_csr()
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self

In [6]:
N = len(FILE_PATHS)
for i, file_path in enumerate(FILE_PATHS):    
    toc = pagerlib_pdf2toc(file_path)
    pred_path = FINTOC_EN_PRED/(file_path.name+'.'+JSON_EXTENSION)
    rez = toc2fintoc_json(toc)
    with open(pred_path, 'w') as f:
        json.dump(rez, f)

    print(f'{i+1}/{N} ({(i+1)/N*100:.2f}%)')

/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


1/10 (10.00%)


/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


2/10 (20.00%)


/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


3/10 (30.00%)


... (10 outputs left out)

/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


9/10 (90.00%)


/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageRStudio/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1787: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


10/10 (100.00%)


# Запуск теста

In [7]:
import subprocess
import pandas as pd

In [8]:
subprocess.run(['python', 'metric.py', '--gt_folder', FINTOC_EN_ANNOTS, '--submission_folder', FINTOC_EN_PRED])

CompletedProcess(args=['python', 'metric.py', '--gt_folder', PosixPath('/home/dataset/fintoc2022/data/en/annots'), '--submission_folder', PosixPath('/home/dataset/fintoc2022/data/en/preds')], returncode=0)

# Вывод результатов

### TD

In [9]:
df = pd.read_csv('td_report.csv', delimiter='\t')
df[-2:-1][['Prec', 'Rec', 'F1']]

,Prec,Rec,F1
280,0.6314012154139478,0.82934210353304,0.7013472670783096


### ToC

In [12]:
df = pd.read_csv('toc_report.csv', delimiter='\t')
tbl = df[-3:-2]
tbl[['Inex08-P', 'Inex08-R', 'Inex08-F1', 'Inex08-Title acc', 'Inex08-Level acc']]

,Inex08-P,Inex08-R,Inex08-F1,Inex08-Title acc,Inex08-Level acc
321,19.5,22.2,20.8,23.1,18.8


In [ ]:
	Prec	Rec	F1
241	0.7648542072526395	0.6965918264951954	0.7003737838531083


In [ ]:
Inex08-P	Inex08-R	Inex08-F1	Inex08-Title acc	Inex08-Level acc
275	44.8	39.6	40.7	52.0	17.2
